<a href="https://colab.research.google.com/github/jrapakko/cs11b/blob/main/%5BCS_11B%5D_Module_3_Practice_Activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Module 3 Practice Activity

**You do not need to submit this Notebook.** It is for practice only.

This Notebook is intended to introduce feature extraction on the famous 20-Newsgroups dataset using the scikit-learn machine learning library, which has several utilities to prepare data for modeling.

### Learn by Experimenting
Please do not simply click "Run all" at the top before reading, as much of the value of this document is to serve as an interactive exploration from one section to the next. In fact, it will help significantly with learning if you first try to understand what the code in a cell might be doing before you run it and get the answer.

Additionally, as you read through the Notebook, you are highly encouraged to change the values in cells to see how the output changes when the cells are re-run. In doing so, you will gain a better understanding of how the various operations behave.

### Notebook Troubleshooting
If you get an unexpected error, the fix is often to click "Run before" from the Runtime menu to run all cells prior to the currently selected cell. If the error persists and you are not sure how to continue, please post on our Canvas forum or send me a message directly with information about the problem.

## Import Modules

In [ ]:
import os

import numpy as np
import pandas as pd

import requests      # Used for web requests
import spacy

from sklearn import datasets
from sklearn import decomposition
from sklearn.feature_extraction import text

In [ ]:
seed = 1

### Install and Import Whisper

OpenAI Whisper will be used to demonstrate automated speech recognition. This library is open source and does not require an API key.

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q
!sudo apt update -qq && sudo apt install ffmpeg -y -q

import whisper

### Audio Recorder Modules

Import the modules below to use your device's microphone to record audio in Google Colab.

If you do not wish to record audio in the next section, you can skip these imports.

In [ ]:
import base64
from google import colab
from IPython import display

### Load the Language Model

In [ ]:
nlp = spacy.load("en_core_web_sm")

---

## Automated Speech Recognition

In this section, we will see Automated Speech Recognition (ASR) in action. In the first part, you can record your voice using your device's microphone, and then see Whisper's attempt to transcribe your words. In the second part, you can download an audio clip from a film and have Whisper transcribe it.

### Record Yourself

The following code will be used to access your device's microphone through Colab. You do not need to run it if you do not wish to use your microphone for the Whisper demonstration.

#### Audio Recording Function Definition

For transparency, AI assisted in the writing of this function.

In [ ]:
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
    const reader = new FileReader()
    reader.onloadend = e => resolve(e.srcElement.result)
    reader.readAsDataURL(blob)
});
var record = time => new Promise(async resolve => {
    stream = await navigator.mediaDevices.getUserMedia({ audio: true })
    recorder = new MediaRecorder(stream)
    chunks = []
    recorder.ondataavailable = e => chunks.push(e.data)
    recorder.start()
    await sleep(time)
    recorder.onstop = async () => {
        blob = new Blob(chunks)
        text = await b2text(blob)
        resolve(text)
    }
    recorder.stop()
});
"""


def record(seconds: int = 5) -> str:
    print("Speak now ...")
    display.Javascript(RECORD)
    try:
        s = colab.output.eval_js(RECORD + f"record({seconds * 1000})")
    except KeyboardInterrupt:
        pass
    b = base64.b64decode(s.split(",")[1])
    recording_path = "audio.wav"
    with open(recording_path, "wb") as f:
        f.write(b)
    print(f"Audio recording saved to 'audio.wav' in this Colab session.")
    return recording_path

#### Start Recording

Run the cell below to begin recording. It should halt after 5 seconds (unless you pass it a different value).

If this cell does not stop after the given number of seconds, the browser may not have granted permissions to your device's microphone properly. Stop the cell and try running it again.

In [ ]:
recording_path = record()

In [ ]:
model = whisper.load_model("base")
model.transcribe(recording_path, fp16=False)["text"]

### Transcribe an Existing Recording

The following process transcribes an existing audio clip.

Unlike the above demonstration, this part does not use your device's microphone.

#### Download the Audio Clip

The small audio file will be stored with your current Colab session.

In [ ]:
movie_quote_path = "./audio.wav"
if not os.path.exists(movie_quote_path):
    url = "https://raw.githubusercontent.com/dkauffman-foothill/media/main/GoodMorning.opus"
    response = requests.get(url)
    with open(movie_quote_path, "wb") as fp:
        fp.write(response.content)

#### Play the Audio Clip

For bragging rights, can you name this movie?

In [ ]:
display.Audio(movie_quote_path)

#### Transcribe the Audio Clip

In [ ]:
model = whisper.load_model("base")
model.transcribe(movie_quote_path, fp16=False)["text"]

---

## Feature Extraction

Now we return to text, and we will be using the famous 20-Newsgroups dataset, comprised of about 18,000 posts on 20 different topics. You can see a list of these topics below.

This corpus is made up of real-world text, and some of the posts may contain objectionable and biased language.

### Newsgroup Categories

- alt.atheism
- comp.graphics
- comp.os.ms-windows.misc
- comp.sys.ibm.pc.hardware
- comp.sys.mac.hardware
- comp.windows.x
- misc.forsale
- rec.autos
- rec.motorcycles
- rec.sport.baseball
- rec.sport.hockey
- sci.crypt
- sci.electronics
- sci.med
- sci.space
- soc.religion.christian
- talk.politics.guns
- talk.politics.mideast
- talk.politics.misc
- talk.religion.misc

### Retrieve Documents

Start by selecting a few topics from the list above, and entering them below (as strings) in the `categories` list. As an example, `sci.space` is included below, but feel free to change it.

You can add as many topics as you like, just know that the more you use, the slower some of the processing steps will become.

In [ ]:
corpus = datasets.fetch_20newsgroups(
    categories=[
        "sci.space",
        "comp.graphics",
    ],
    remove=("headers", "footers", "quotes")
)["data"]

len(corpus)

We can inspect one of the posts from the corpus by indexing on it.

In [ ]:
corpus[0]

#### Define the Tokenizer

This spaCy-based tokenizer will be used in the vectorization methods below to break up text, which provides more control over using scikit-learn's built-in tokenizer.

Feel free to make changes to this tokenizer, such as which parts of speech are retained.

In [ ]:
def spacy_tokenizer(text: str) -> list[str]:
    doc = nlp(text)
    return [
        token.lemma_ for token in doc
        if (
            token.is_alpha
            and not token.is_stop
            and token.pos_ in ["NOUN", "VERB", "ADJ", "ADV"]
        )
    ]

### Bag of Words

Let's create our first matrix using `CountVectorizer`, which uses the Bag-of-Words approach. The tokenizer defined above will be used to create the tokens that will be transformed into n-grams for the vocabulary.

Try experimenting with different ranges of n-grams.

In [ ]:
bow_transformer = text.CountVectorizer(
    tokenizer=spacy_tokenizer,
    ngram_range=(2, 4),
    token_pattern=None
)

bow_matrix = bow_transformer.fit_transform(corpus)
bow_matrix.shape

We can convert this Bag-of-Words into a DataFrame, add up the count of each n-gram across all documents, and look at the 50 most frequent. Note that doing so eliminates the memory savings of using a sparse array, but can be useful for inspecting data.

In [ ]:
df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_transformer.get_feature_names_out()
)

df.shape

In [ ]:
df.sum().rename("count").sort_values(ascending=False).head(50)

### Term Frequency Inverse Document Frequency (TF-IDF)

To create weighted features using TF-IDF, we can use the transformer below. Here, we do not use a tokenizer or generate n-grams, as this matrix will be used as a baseline for a later section.

In [ ]:
tfidf_transformer = text.TfidfVectorizer()
tfidf_matrix = tfidf_transformer.fit_transform(corpus)
tfidf_matrix.shape

Again, we convert the matrix into a DataFrame.

In [ ]:
df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_transformer.get_feature_names_out()
)

df.shape

Notice that the values are no longer integers. See Canvas for a breakdown of how these TF-IDF weights are computed.

In [ ]:
df.sum().rename("weight").sort_values(ascending=False).head(50)

---

## Feature Reduction

Because the matrices created by these vectorization processes are often very large, we can get much better performance by reducing their dimensionality - that is, the number of columns in the matrix - while still retaining much of their explanatory power.

Below, we an NLP known as Latent Semantic Analysis (LSA) with the `TruncatedSVD` transformer to reduce the number of columns to 100 - as set by `n_components` below.

Upcoming Canvas content will explain this process in more detail.

In [ ]:
svd_transformer = decomposition.TruncatedSVD(n_components=100, random_state=seed)
svd_matrix = svd_transformer.fit_transform(tfidf_matrix)
svd_matrix.shape

We can see how much information was retained by looking at the explained variance ratio, where a value of 1 means all information was retained. You can experiment with higher values for `n_components` above to find a good balance between more information retained and a manageable matrix size.

In [ ]:
svd_transformer.explained_variance_ratio_.sum()

## Comparison of Extraction Methods

In this section, we will try three feature extraction methods for comparison.

1. Using TF-IDF with a spaCy tokenizer
2. Using TF-IDF with n-grams and vocabulary pruning (no spaCy tokenizer)
3. Using both methods above

We will also compare them with the TF-IDF matrix created in the previous section, which will serve as a baseline.

Let's start by renaming the transformers defined in previous sections more appropriately.

In [ ]:
baseline_tfidf = tfidf_transformer
baseline_svd = svd_transformer

### Method 1: Using spaCy Preprocessing

To begin, we will try TF-IDF with the spaCy tokenizer defined earlier, but no n-grams or vocabulary pruning.

Note that `token_pattern=None` is only used to suppress a warning here, and is not strictly necessary.

In [ ]:
m1_tfidf = text.TfidfVectorizer(
    tokenizer=spacy_tokenizer,
    token_pattern=None
)

m1_matrix = m1_tfidf.fit_transform(corpus)
m1_matrix.shape

In [ ]:
m1_svd = decomposition.TruncatedSVD(n_components=100, random_state=seed)
m1_svd_matrix = m1_svd.fit_transform(m1_matrix)
m1_svd_matrix.shape

In [ ]:
m1_svd.explained_variance_ratio_.sum()

### Method 2: Using Vocabulary Pruning

Next, we will forgo our spaCy tokenizer and apply vocabulary pruning with the `min_df` and `max_df` parameters of the vectorizer. If assigned an integer, it represents a number of documents in which an n-gram can appear; if assigned a float, it represents a percentage of documents in which an n-gram can appear. This pruning helps to remove very common and very rare words that would usually not be helpful for an analysis.

Example parameter arguments:
- Use `min_df=5` to remove words that appear in fewer than 5 documents
- Use `max_df=0.7` to remove words that appear in more than 70% of documents

In addition, you can set which n-grams to add to the vocabulary with `ngram_range`, where the first integer represents the lowest order n-grams created, and the second integer represents the highest order n-grams created.

In [ ]:
m2_tfidf = text.TfidfVectorizer(
    min_df=5,
    max_df=0.7,
    ngram_range=(1, 2)
)
m2_matrix = m2_tfidf.fit_transform(corpus)
m2_matrix.shape

In [ ]:
m2_svd = decomposition.TruncatedSVD(n_components=100, random_state=seed)
m2_svd_matrix = m2_svd.fit_transform(m2_matrix)
m2_svd_matrix.shape

In [ ]:
m2_svd.explained_variance_ratio_.sum()

### Method 3: Combining Methods 1 and 2

In the final method, we combine the previous two approaches.

In [ ]:
m3_tfidf = text.TfidfVectorizer(
    tokenizer=spacy_tokenizer,
    min_df=5,
    max_df=0.7,
    token_pattern=None
)

m3_matrix = m3_tfidf.fit_transform(corpus)
m3_matrix.shape

In [ ]:
m3_svd = decomposition.TruncatedSVD(n_components=100, random_state=seed)
m3_svd_matrix = m3_svd.fit_transform(m3_matrix)
m3_svd_matrix.shape

In [ ]:
m3_svd.explained_variance_ratio_.sum()

### Comparing Methodologies

Let's inspect the sizes of the vocabularies created for each method (which includes n-grams), and the explained variance ratio after applying dimensionality reduction with LSA.

In [ ]:
pd.DataFrame({
    "Approach": [
        "Baseline",
        "Method 1",
        "Method 2",
        "Method 3"
    ],
    "Vocabulary Size": [
        len(baseline_tfidf.vocabulary_),
        len(m1_tfidf.vocabulary_),
        len(m2_tfidf.vocabulary_),
        len(m3_tfidf.vocabulary_)
    ],
    "Explained Variance": [
        baseline_svd.explained_variance_ratio_.sum(),
        m1_svd.explained_variance_ratio_.sum(),
        m2_svd.explained_variance_ratio_.sum(),
        m3_svd.explained_variance_ratio_.sum()
    ]
})

---

## Next Steps

The project for this module will have you employ these feature engineering steps across a corpus of many documents. See the Project Check-In survey in Canvas for more information.